## 4_relabeling_examples.ipynb
### Fairness and Compliance of AI
### University of Oklahoma

Relabeling addresses systematic unfairness. We conduct a baseline regression based upon true labels to determine the probability of being assigned to the positive label class. We sort our training data set by probabilities to relabel negative observations that are close to the decision boundary until we have demographic parity in the training set. We then use the relabeled training data to run a new classification.

We use the UCI Adult Census dataset for which we imputed missing values for the analysis. Seeds are explicitly set to ensure identical results across runs and machines.

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, mean_absolute_error, \
     mean_absolute_percentage_error, mean_squared_error
import os, random

# Set global seeds for reproducibility
os.environ["PYTHONHASHSEED"] = "42"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
random.seed(42)
np.random.seed(42)


In [ ]:
# ---------------------------------------------------------
# 1. RELABELING FOR FAIRNESS
# ---------------------------------------------------------

# ---------------------------------------------------------
# 1.1. Load Adult dataset and encode labels
# ---------------------------------------------------------
df = pd.read_csv("adult_imputed.csv")

# Protected attribute (A) and label (Y)
A = df["sex"]
Y_class = df["target"]

# Encode protected attribute and label for weighting + evaluation
A_le = LabelEncoder()
Y_le = LabelEncoder()
A_enc = A_le.fit_transform(A)
Y_class_enc = Y_le.fit_transform(Y_class)

# ---------------------------------------------------------
# Build predictor matrix and transform
# ---------------------------------------------------------
X = df.drop(columns=["target"])   # <-- sex stays in the predictors

# Split numeric and categorical
numeric_cols = X.select_dtypes(include=[np.number]).columns
categorical_cols = X.select_dtypes(exclude=[np.number]).columns

# Z-score normalize numeric predictors
scaler = StandardScaler()
X_numeric_scaled = pd.DataFrame(
    scaler.fit_transform(X[numeric_cols]),
    columns=numeric_cols,
    index=X.index
)

# One-hot encode categorical predictors (including 'sex')
X_categorical = pd.get_dummies(X[categorical_cols], drop_first=True)

# Combine into final feature matrix
X_enc = pd.concat([X_numeric_scaled, X_categorical], axis=1)

### Run the baseline regression model to get prediction probabilities

In [ ]:
# ---------------------------------------------------------
# 1.2. Run the baseline regression model
# ---------------------------------------------------------
X_train, X_test, y_train, y_test, A_train, A_test = train_test_split(
    X_enc, Y_class_enc, A_enc, test_size=0.3, random_state=42
)

clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# Prediction probabilities on training set
probs = clf.predict_proba(X_train)[:, 1]


### Relabel negative observations for the disadvantaged group

In [ ]:
# ---------------------------------------------------------
# 3. Threshold-Based Relabeling
# ---------------------------------------------------------
# Goal: Improve demographic parity by flipping labels for the disadvantaged group
# Example: Suppose group A=0 is disadvantaged (e.g., Female)

disadvantaged = 0

# Identify negative labels in disadvantaged group
mask = (A_train == disadvantaged) & (y_train == 0)

# Sort by model confidence (highest probability of being positive)
borderline_idx = np.argsort(probs[mask])[::-1]

# Compute demographic parity gap BEFORE relabeling
p1 = np.mean(y_train[A_train == 1])
p0 = np.mean(y_train[A_train == 0])
gap_before = p0 - p1
print("Demographic Parity Gap BEFORE relabeling:", gap_before)

# Flip labels until parity improves
y_train_relab = y_train.copy()
target_gap = 0.02  # desired gap threshold

for idx in np.where(mask)[0][borderline_idx]:
    y_train_relab[idx] = 1  # flip label
    p1_new = np.mean(y_train_relab[A_train == 1])
    p0_new = np.mean(y_train_relab[A_train == 0])
    if abs(p0_new - p1_new) <= target_gap:
        break

print("Demographic Parity Gap AFTER relabeling:", p0_new - p1_new)

### Retrain the model on the relabeled data

In [ ]:
# ---------------------------------------------------------
# 4. Retrain model on relabeled data
# ---------------------------------------------------------
clf_relab = LogisticRegression(max_iter=200)
clf_relab.fit(X_train, y_train_relab)

### Compare results
We compare the performance of the baseline and relabeled regressions. We look at summary performance (f1-score), the distribution of income predictions, and error-rate parity.

In [ ]:
# ---------------------------------------------------------
# 5. Compare regression summary results
# ---------------------------------------------------------
print("\n=== Original Model ===")
print(classification_report(y_test, clf.predict(X_test)))

print("\n=== Relabeled Model ===")
print(classification_report(y_test, clf_relab.predict(X_test)))

In [ ]:
# ---------------------------------------------------------
# 6. Prediction vs Sex Tables
# ---------------------------------------------------------

# Predictions
pred_orig = clf.predict(X_test)
pred_relab = clf_relab.predict(X_test)

# Decode sex for readability
sex_test_decoded = A_le.inverse_transform(A_test)

# Build DataFrames
df_orig = pd.DataFrame({
    "sex": sex_test_decoded,
    "prediction": Y_le.inverse_transform(pred_orig)
})

df_relab = pd.DataFrame({
    "sex": sex_test_decoded,
    "prediction": Y_le.inverse_transform(pred_relab)
})

print("\n=== Original Model: Prediction × Sex ===")
print(pd.crosstab(df_orig["sex"], df_orig["prediction"]))

print("\n=== Relabeled Model: Prediction × Sex ===")
print(pd.crosstab(df_relab["sex"], df_relab["prediction"]))


In [ ]:
# ---------------------------------------------------------
# 7. Classification Error-Rate Parity Test (Separation)
# ---------------------------------------------------------

def error_rates_by_group(y_true, y_pred, A_test, A_encoder, Y_encoder):
    """
    Calculate False Positive Rate and False Negative Rate by protected group. This function
    uses the encoded protected attribute values and label values, but decodes them in the output
    for readability.

    parameters:
    - y_true: true labels
    - y_pred: predicted labels
    - A_test: protected attribute values for test set
    - A_encoder: LabelEncoder for protected attribute (to decode group names)
    - Y_encoder: LabelEncoder for label (to decode label names)

    returns:
    - DataFrame with FPR and FNR for each protected group
    """
    
    results = []

    for group in np.unique(A_test):
        mask = (A_test == group)
        yt = y_true[mask]
        yp = y_pred[mask]

        # False Positive Rate: FP / (FP + TN)
        fp = np.sum((yp == 1) & (yt == 0))
        tn = np.sum((yp == 0) & (yt == 0))
        fpr = fp / (fp + tn) if (fp + tn) > 0 else np.nan

        # False Negative Rate: FN / (FN + TP)
        fn = np.sum((yp == 0) & (yt == 1))
        tp = np.sum((yp == 1) & (yt == 1))
        fnr = fn / (fn + tp) if (fn + tp) > 0 else np.nan

        results.append({
            "sex": A_encoder.inverse_transform([group])[0],
            "FPR (predicting {})".format(Y_encoder.inverse_transform([1])[0]): round(fpr, 3),
            "FNR (predicting {})".format(Y_encoder.inverse_transform([1])[0]): round(fnr, 3)
        })

    return pd.DataFrame(results)


# Extract protected attribute for the test split
_, A_test = train_test_split(A_enc, test_size=0.3, random_state=42)

pred_baseline = clf.predict(X_test)
pred_relabeled = clf_relab.predict(X_test)

print("\n=== Error-Rate Parity: Baseline Model ===")
print(error_rates_by_group(y_test, pred_baseline, A_test, A_le, Y_le))

print("\n=== Error-Rate Parity: Relabeled Model ===")
print(error_rates_by_group(y_test, pred_relabeled, A_test, A_le, Y_le))

